In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CNN Model
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(CNN, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x


def train_model(dataset_name, dataset, in_channels, num_classes):

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64)

    model = CNN(in_channels, num_classes).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_acc = 0

    for epoch in range(10):

        model.train()
        total_loss = 0

        for x, y in train_loader:

            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            output = model(x)

            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # Validation
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for x, y in val_loader:

                x, y = x.to(device), y.to(device)

                output = model(x)
                _, pred = torch.max(output, 1)

                total += y.size(0)
                correct += (pred == y).sum().item()

        val_acc = correct / total

        print(f"{dataset_name} Epoch {epoch+1} | Loss {total_loss:.4f} | Val Acc {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f"{dataset_name}_best.pth")
            print("Checkpoint saved")


In [22]:
transform_mnist = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor()
])

transform_cifar = transforms.Compose([
    transforms.ToTensor()
])

mnist = datasets.MNIST(root='./data', train=True, download=True, transform=transform_mnist)

cifar10 = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)

cifar100 = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_cifar)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:05<00:00, 1.89MB/s]


Extracting ./data\MNIST\raw\train-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 64.7kB/s]


Extracting ./data\MNIST\raw\train-labels-idx1-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



...

Extracting ./data\MNIST\raw\t10k-labels-idx1-ubyte.gz to ./data\MNIST\raw



100%|██████████| 170M/170M [00:50<00:00, 3.37MB/s]   


Extracting ./data\cifar-10-python.tar.gz to ./data


100%|██████████| 169M/169M [00:20<00:00, 8.10MB/s] 


Extracting ./data\cifar-100-python.tar.gz to ./data


In [23]:
train_model("MNIST", mnist, 1, 10)

train_model("CIFAR10", cifar10, 3, 10)

train_model("CIFAR100", cifar100, 3, 100)

MNIST Epoch 1 | Loss 154.1810 | Val Acc 0.9826
Checkpoint saved
MNIST Epoch 2 | Loss 39.7387 | Val Acc 0.9861
Checkpoint saved
MNIST Epoch 3 | Loss 27.6018 | Val Acc 0.9866
Checkpoint saved
MNIST Epoch 4 | Loss 20.2998 | Val Acc 0.9888
Checkpoint saved
MNIST Epoch 5 | Loss 17.6496 | Val Acc 0.9886
MNIST Epoch 6 | Loss 13.4111 | Val Acc 0.9902
Checkpoint saved
MNIST Epoch 7 | Loss 11.1464 | Val Acc 0.9886
MNIST Epoch 8 | Loss 9.7724 | Val Acc 0.9905
Checkpoint saved
MNIST Epoch 9 | Loss 8.1183 | Val Acc 0.9912
Checkpoint saved
MNIST Epoch 10 | Loss 7.3730 | Val Acc 0.9888
CIFAR10 Epoch 1 | Loss 982.4656 | Val Acc 0.5092
Checkpoint saved
CIFAR10 Epoch 2 | Loss 734.6379 | Val Acc 0.6036
Checkpoint saved
CIFAR10 Epoch 3 | Loss 617.2227 | Val Acc 0.6630
Checkpoint saved
CIFAR10 Epoch 4 | Loss 534.8796 | Val Acc 0.6564
CIFAR10 Epoch 5 | Loss 473.1557 | Val Acc 0.6977
Checkpoint saved
CIFAR10 Epoch 6 | Loss 422.9883 | Val Acc 0.7090
Checkpoint saved
CIFAR10 Epoch 7 | Loss 373.2145 | Val Acc 0

In [24]:
def evaluate(dataset_name, test_dataset, in_channels, num_classes):

    loader = DataLoader(test_dataset, batch_size=64)

    model = CNN(in_channels, num_classes).to(device)
    model.load_state_dict(torch.load(f"{dataset_name}_best.pth"))

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in loader:

            x, y = x.to(device), y.to(device)

            output = model(x)

            _, pred = torch.max(output, 1)

            total += y.size(0)
            correct += (pred == y).sum().item()

    acc = correct / total

    print(f"{dataset_name} Test Accuracy: {acc:.4f}")

In [25]:
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform_mnist)

cifar10_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar)

cifar100_test = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_cifar)

evaluate("MNIST", mnist_test, 1, 10)
evaluate("CIFAR10", cifar10_test, 3, 10)
evaluate("CIFAR100", cifar100_test, 3, 100)

Files already downloaded and verified
Files already downloaded and verified


C:\Users\HARSH\AppData\Local\Temp\ipykernel_19336\2551082931.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"{dataset_name}_best.pth")

MNIST Test Accuracy: 0.9917
CIFAR10 Test Accuracy: 0.7173
CIFAR100 Test Accuracy: 0.3825
